In [ ]:
# DFU Phase-4 post-hoc: no training, no CNN inference, saved predictions only
import urllib.request, hashlib, base64, zlib

VERSION = "DFU_PHASE4_POSTHOC_NO_INFERENCE_LOADER_V1_20260812"
SOURCE_COMMIT = "9f0d9770105b099d5864abf1d42fc6ae36224317"
PARTS = [
    ("scripts/phase4_posthoc_v1_payload/part_00.txt", "66fc57edf9a2043ef49e4724bdc265a9104cfedc"),
    ("scripts/phase4_posthoc_v1_payload/part_01.txt", "9e9c58b3321d790c6e10aca568d61e2a265a1cf0"),
]
EXPECTED_PAYLOAD_SHA256 = "acafb61a896758a7762f97bf6e037f8fff05a329c619e9039f165de436740036"
EXPECTED_SOURCE_SHA256 = "32360f06f76091c51ddea2465863cda114fa29328098e2b27dab09636212d1ec"
BASE = f"https://raw.githubusercontent.com/AzizulHakim00/DFU-ImageGuard/{SOURCE_COMMIT}/"

def git_blob_sha(raw):
    return hashlib.sha1(b"blob " + str(len(raw)).encode() + b"\0" + raw).hexdigest()

print("="*100)
print(VERSION)
print("POST-HOC ONLY | NO TRAINING | NO CNN INFERENCE | EXISTING SAVED PREDICTIONS ONLY")
print("="*100)
chunks=[]
for path, expected_blob in PARTS:
    raw=urllib.request.urlopen(BASE+path, timeout=120).read()
    actual=git_blob_sha(raw)
    if actual != expected_blob:
        raise RuntimeError(f"Payload fragment mismatch: {path} expected={expected_blob} actual={actual}")
    print("Payload fragment PASS:", path, actual)
    chunks.append(raw)
payload=b"".join(chunks)
payload_sha=hashlib.sha256(payload).hexdigest()
if payload_sha != EXPECTED_PAYLOAD_SHA256:
    raise RuntimeError(f"Payload SHA mismatch: expected={EXPECTED_PAYLOAD_SHA256} actual={payload_sha}")
print("Combined payload SHA256: PASS", payload_sha)
source=zlib.decompress(base64.b64decode(payload, validate=True))
source_sha=hashlib.sha256(source).hexdigest()
if source_sha != EXPECTED_SOURCE_SHA256:
    raise RuntimeError(f"Decoded source SHA mismatch: expected={EXPECTED_SOURCE_SHA256} actual={source_sha}")
print("Decoded source SHA256: PASS", source_sha)
text=source.decode("utf-8")
compile(text, "dfu_phase4_posthoc_v1.py", "exec")
print("Decoded source compile: PASS")
print("Starting post-hoc analysis...")
exec(compile(text, "dfu_phase4_posthoc_v1.py", "exec"), globals())
